# LegalBERT: policy-grouped five-fold threshold experiment

Run all cells in a GPU runtime. Five fold models are trained on the original 65 training policies.
Each policy is held out once, and thresholds are selected only from the combined out-of-fold probabilities.
The selected global and per-category thresholds are then applied to the untouched 50-policy evaluation set
for every completed main-study seed. Fold-level completion receipts allow safe restart after disconnection.


In [ ]:
from pathlib import Path, PurePosixPath
import hashlib, json, subprocess, sys, zipfile
from google.colab import drive, files
drive.mount('/content/drive')
MODEL = 'LegalBERT'
ARCHIVE_NAME = 'LegalBERT_threshold_cv_workspace.zip'
EXPECTED_ARCHIVE_SHA256 = '8827361d2f71d185b1d4ea14db166d164a703d2d6a628986973b6ff7bb50c020'
archive_path = Path('/content/drive/MyDrive/OPP115_ThresholdCV/packages') / ARCHIVE_NAME
existing_digest = hashlib.sha256(archive_path.read_bytes()).hexdigest() if archive_path.is_file() else None
if existing_digest != EXPECTED_ARCHIVE_SHA256:
    if existing_digest is not None:
        print('The Drive copy is an older package and must be replaced:', archive_path)
    print('Upload', ARCHIVE_NAME)
    uploaded = files.upload()
    if set(uploaded) != {ARCHIVE_NAME}:
        raise ValueError('Upload exactly ' + ARCHIVE_NAME)
    archive_path.parent.mkdir(parents=True, exist_ok=True)
    archive_path.write_bytes(uploaded[ARCHIVE_NAME])
archive_digest = hashlib.sha256(archive_path.read_bytes()).hexdigest()
PROJECT_DIR = Path('/content') / (MODEL + '_threshold_cv_' + archive_digest[:12])
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(archive_path) as archive:
    if len(archive.infolist()) > 5000 or sum(x.file_size for x in archive.infolist()) > 2 * 1024**3:
        raise ValueError('Archive exceeds workspace limits')
    for entry in archive.infolist():
        member = PurePosixPath(entry.filename)
        if member.is_absolute() or '..' in member.parts or ((entry.external_attr >> 16) & 0o170000) == 0o120000:
            raise ValueError('Unsafe archive member')
    archive.extractall(PROJECT_DIR)
manifest = json.loads((PROJECT_DIR / 'threshold_cv_workspace.manifest.json').read_text())
for item in manifest['files']:
    path = PROJECT_DIR / item['path']
    if not path.is_file() or path.stat().st_size != item['bytes'] or hashlib.sha256(path.read_bytes()).hexdigest() != item['sha256']:
        raise ValueError('Workspace verification failed: ' + item['path'])
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT_DIR / 'requirements_colab_legal_bert_opp115.txt')], check=True)
print('Verified workspace:', PROJECT_DIR)


In [ ]:
MODE = 'full'  #@param ['full', 'smoke']
STUDY_NAME = 'policy_5fold_thresholds_v1'  #@param {type:'string'}
OUTPUT_ROOT = Path('/content/drive/MyDrive/OPP115_ThresholdCV/studies') / STUDY_NAME
MAIN_STUDY_ROOT = Path('/content/drive/MyDrive/OPP115_MultiSeed/studies/matched_five_seeds_v1')
import torch
if MODE == 'full' and not torch.cuda.is_available():
    raise RuntimeError('Select Runtime > Change runtime type > GPU, reconnect, and run all cells.')
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU smoke mode')
print('Threshold study:', OUTPUT_ROOT)
print('Completed main-study checkpoints:', MAIN_STUDY_ROOT)


In [ ]:
command = [sys.executable, '-u', str(PROJECT_DIR / 'threshold_cv.py'),
           '--model-dir', str(PROJECT_DIR), '--data-dir', str(PROJECT_DIR / 'data/opp115'),
           '--output-root', str(OUTPUT_ROOT), '--main-study-root', str(MAIN_STUDY_ROOT),
           '--mode', MODE]
log_path = OUTPUT_ROOT / MODE / MODEL / 'execution.log'
log_path.parent.mkdir(parents=True, exist_ok=True)
print('Live execution log:', log_path)
with log_path.open('a', encoding='utf-8') as log:
    log.write('\nCOMMAND: ' + ' '.join(command) + '\n')
    process = subprocess.Popen(command, cwd=PROJECT_DIR, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='')
        log.write(line)
        log.flush()
    return_code = process.wait()
if return_code:
    raise RuntimeError(
        f'Threshold experiment exited with code {return_code}. '
        f'The complete underlying error is printed above and saved at {log_path}.'
    )


In [ ]:
from IPython.display import Markdown, display
report_path = OUTPUT_ROOT / MODE / MODEL / 'report.md'
display(Markdown(report_path.read_text()))
print('Selected thresholds:', report_path.parent / 'selected_thresholds.json')
print('Held-out results:', report_path.parent / 'heldout_results.json')


In [ ]:
models = ['BERT', 'RoBERTa', 'LegalBERT', 'PrivBERT']
missing = [model for model in models if not (OUTPUT_ROOT / 'full' / model / 'heldout_results.json').is_file()]
if MODE != 'full':
    print('Smoke mode validates execution only.')
elif missing:
    print('Complete these model notebooks before comparison:', ', '.join(missing))
else:
    subprocess.run([sys.executable, str(PROJECT_DIR / 'compare_thresholds.py'),
                    '--output-root', str(OUTPUT_ROOT)], check=True)
    display(Markdown((OUTPUT_ROOT / 'full/threshold_comparison.md').read_text()))
